In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
import requests
import os
from datetime import date, timedelta, datetime

## Part 1

The source of this data comes from NASA's NeoWs (Near Earth Object Web Service). When requesting data from the API, the end data can be a maximum of seven days after the start date. The raw data contains lots of information about asteriods that are considered Near Earth Objects (NEOs). In this project, only seven variable from the raw data are considered, these seven variables include: Approach Date, Miss Distance, Relative Velocity, Object Id (primary key), Absolute Magnitude, Estimated Diameter, and Potentially Hazardous Asteriod flag.

Every approach date contians approach data from an object on that day. Because of this reason, the data is read in as two seperate DataFrames that are merged on approach date. Approach date is used as the key for this merge, however, moving forward, Object Id is used as the primary key as it is more practical.

The first DataFrame created contains the Approach Date, Miss Distance, and Relative Velocity for every NEO observed on that day. The second DataFrame contains the Approach Date, Object Id, Absolute Magnitude, Estimated Diameter, and Potentially Hazardous Asteriod flag. The final dataset containing all NEO observations is titled neos_df.

The neos_df contains 7 columns, 480 rows, and no missing values. The full structure of the DataFrame is as follows:
<pre>
 #   Column                             Non-Null Count  Dtype  
 0   Approach Date                      480 non-null    datetime64[us]
 1   miss_distance                      480 non-null    float64       
 2   relative_velocity                  480 non-null    float64       
 3   object_id                          480 non-null    str           
 4   absolute_magnitude_h               480 non-null    float64       
 5   estimated_diameter                 480 non-null    float64       
 6   is_potentially_hazardous_asteroid  480 non-null    bool
 <pre>

In [2]:
url = "https://api.nasa.gov/neo/rest/v1/feed"

SESSION = requests.Session()

response = requests.get(url, timeout=10,
                        params={"api_key": "DEMO_KEY",
                                "start_date": "2026-10-21",
                                "end_date": "2026-10-28"})

In [3]:
data = response.json()
print(data.keys())

neos = data['near_earth_objects']

dict_keys(['element_count', 'links', 'near_earth_objects'])


In [4]:
neos.keys()

dict_keys(['2026-10-28', '2026-10-27', '2026-10-22', '2026-10-21', '2026-10-26', '2026-10-25', '2026-10-24', '2026-10-23'])

In [5]:
neos['2026-10-28'][0]

{'absolute_magnitude_h': 21.85,
 'close_approach_data': [{'close_approach_date': '2026-10-28',
   'close_approach_date_full': '2026-Oct-28 16:55',
   'epoch_date_close_approach': 1793206500000,
   'miss_distance': {'astronomical': '0.1530448084',
    'kilometers': '22895177.351198108',
    'lunar': '59.5344304676',
    'miles': '14226403.5189375704'},
   'orbiting_body': 'Earth',
   'relative_velocity': {'kilometers_per_hour': '20345.8248160687',
    'kilometers_per_second': '5.6516180045',
    'miles_per_hour': '12642.1043141844'}}],
 'estimated_diameter': {'feet': {'estimated_diameter_max': 831.8117101206,
   'estimated_diameter_min': 371.997505662},
  'kilometers': {'estimated_diameter_max': 0.2535362011,
   'estimated_diameter_min': 0.1133848361},
  'meters': {'estimated_diameter_max': 253.5362011316,
   'estimated_diameter_min': 113.3848360975},
  'miles': {'estimated_diameter_max': 0.1575400428,
   'estimated_diameter_min': 0.070454049}},
 'id': '2518810',
 'is_potentially_hazard

In [6]:
neos['2026-10-28'][0]['close_approach_data']

[{'close_approach_date': '2026-10-28',
  'close_approach_date_full': '2026-Oct-28 16:55',
  'epoch_date_close_approach': 1793206500000,
  'miss_distance': {'astronomical': '0.1530448084',
   'kilometers': '22895177.351198108',
   'lunar': '59.5344304676',
   'miles': '14226403.5189375704'},
  'orbiting_body': 'Earth',
  'relative_velocity': {'kilometers_per_hour': '20345.8248160687',
   'kilometers_per_second': '5.6516180045',
   'miles_per_hour': '12642.1043141844'}}]

In [7]:
approach_dates = [app_date for app_date in neos]

close_approach_data = {
    "Approach Date": [neos[ad][i]['close_approach_data'][j]['close_approach_date']
              for ad in approach_dates for i in range(len(neos[ad])) 
              for j in range(len(neos[ad][i]['close_approach_data']))],
    
    "miss_distance": [float(neos[ad][i]['close_approach_data'][j]['miss_distance']['kilometers']) 
              for ad in approach_dates for i in range(len(neos[ad])) 
              for j in range(len(neos[ad][i]['close_approach_data']))],

    "relative_velocity": [float(neos[ad][i]['close_approach_data'][j]['relative_velocity']['kilometers_per_hour']) 
              for ad in approach_dates for i in range(len(neos[ad])) 
              for j in range(len(neos[ad][i]['close_approach_data']))],
}

close_approach_df = pd.DataFrame(data=close_approach_data)

In [8]:
close_approach_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Approach Date      60 non-null     str    
 1   miss_distance      60 non-null     float64
 2   relative_velocity  60 non-null     float64
dtypes: float64(2), str(1)
memory usage: 1.5 KB


In [9]:
object_data = {
    "Approach Date": [ad for ad in neos for _ in range(len(neos[ad]))],

    "object_id": [neos[ad][i]['id'] for ad in approach_dates for i in range(len(neos[ad]))],

    "absolute_magnitude_h": [neos[ad][i]['absolute_magnitude_h'] for ad in approach_dates for i in range(len(neos[ad]))],
    
    "estimated_diameter": [neos[ad][i]['estimated_diameter']['kilometers']['estimated_diameter_min'] 
                           for ad in approach_dates
                              for i in range(len(neos[ad]))],

    "is_potentially_hazardous_asteroid": [neos[ad][i]['is_potentially_hazardous_asteroid']
                                          for ad in approach_dates
                                            for i in range(len(neos[ad]))],
}

object_data_df = pd.DataFrame(data=object_data)

In [10]:
object_data_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 5 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   Approach Date                      60 non-null     str    
 1   object_id                          60 non-null     str    
 2   absolute_magnitude_h               60 non-null     float64
 3   estimated_diameter                 60 non-null     float64
 4   is_potentially_hazardous_asteroid  60 non-null     bool   
dtypes: bool(1), float64(2), str(2)
memory usage: 2.1 KB


In [11]:
neos_df = close_approach_df.merge(object_data_df, on="Approach Date", how="left", validate="many_to_many")
neos_df['Approach Date'] = pd.to_datetime(neos_df['Approach Date'])
neos_df = neos_df.sort_values(by='object_id')
display(neos_df.head())

,Approach Date,miss_distance,relative_velocity,object_id,absolute_magnitude_h,estimated_diameter,is_potentially_hazardous_asteroid
210,2026-10-26,6.642528e+07,67073.229700,2488509,20.61,0.200704,False
282,2026-10-26,4.635502e+07,32270.110647,2488509,20.61,0.200704,False
273,2026-10-26,4.509338e+07,73474.264537,2488509,20.61,0.200704,False
264,2026-10-26,2.116466e+07,12565.295839,2488509,20.61,0.200704,False
246,2026-10-26,6.486772e+07,71372.128055,2488509,20.61,0.200704,False


In [12]:
neos_df.info()

<class 'pandas.DataFrame'>
Index: 480 entries, 210 to 340
Data columns (total 7 columns):
 #   Column                             Non-Null Count  Dtype         
---  ------                             --------------  -----         
 0   Approach Date                      480 non-null    datetime64[us]
 1   miss_distance                      480 non-null    float64       
 2   relative_velocity                  480 non-null    float64       
 3   object_id                          480 non-null    str           
 4   absolute_magnitude_h               480 non-null    float64       
 5   estimated_diameter                 480 non-null    float64       
 6   is_potentially_hazardous_asteroid  480 non-null    bool          
dtypes: bool(1), datetime64[us](1), float64(4), str(1)
memory usage: 26.7 KB


## Part 2

In [13]:
import sys
from pathlib import Path

sys.path.append(str(Path("../src").resolve()))

import neo_functions as nf

In [14]:
hazardous_neos = nf.group_neos_by_hazardous_flag(df=neos_df, hazardous_flag=True)
nonhazardous_neos = nf.group_neos_by_hazardous_flag(df=neos_df, hazardous_flag=False)

In [15]:
display(hazardous_neos.head())
display(nonhazardous_neos.head())

,Approach Date,miss_distance,relative_velocity,object_id,absolute_magnitude_h,estimated_diameter,is_potentially_hazardous_asteroid
30,2026-10-28,5.848717e+07,52895.829085,2518810,21.85,0.113385,True
24,2026-10-28,4.240878e+06,33549.810808,2518810,21.85,0.113385,True
0,2026-10-28,2.289518e+07,20345.824816,2518810,21.85,0.113385,True
6,2026-10-28,3.383045e+07,52334.090119,2518810,21.85,0.113385,True
12,2026-10-28,6.556904e+07,40395.479196,2518810,21.85,0.113385,True


,Approach Date,miss_distance,relative_velocity,object_id,absolute_magnitude_h,estimated_diameter,is_potentially_hazardous_asteroid
210,2026-10-26,6.642528e+07,67073.229700,2488509,20.61,0.200704,False
282,2026-10-26,4.635502e+07,32270.110647,2488509,20.61,0.200704,False
273,2026-10-26,4.509338e+07,73474.264537,2488509,20.61,0.200704,False
264,2026-10-26,2.116466e+07,12565.295839,2488509,20.61,0.200704,False
246,2026-10-26,6.486772e+07,71372.128055,2488509,20.61,0.200704,False


In [16]:
filtered_hazardous_neos = nf.filter_by_date(df=hazardous_neos, start_date="2026-10-21", end_date="2026-10-22")
display(filtered_hazardous_neos.head())

,Approach Date,miss_distance,relative_velocity,object_id,absolute_magnitude_h,estimated_diameter,is_potentially_hazardous_asteroid
140,2026-10-21,5.556045e+07,28776.250761,3387458,21.88,0.111829,True
130,2026-10-21,5.737689e+07,67295.922468,3387458,21.88,0.111829,True
120,2026-10-21,3.029499e+07,50814.451712,3387458,21.88,0.111829,True
180,2026-10-21,3.387593e+07,27858.622741,3387458,21.88,0.111829,True
150,2026-10-21,4.628238e+07,46586.495612,3387458,21.88,0.111829,True


In [17]:
hazardous_neos_summary, nonhazardous_neos_summary = nf.summarize_potentialy_hazardous_neos(df=neos_df)
display(hazardous_neos_summary.head())
display(nonhazardous_neos_summary.head())

,n_approaches,miss_min,miss_median,miss_max,vel_min,vel_median,vel_max,h,diameter,hazardous
object_id,,,,,,,,,,
2518810,6,4240878.05,40771124.53,65569040.72,20345.82,45495.14,52895.83,21.85,0.11,True
3387458,10,30294985.22,46741360.34,59698211.11,27858.62,58049.68,132950.77,21.88,0.11,True
3753695,10,30294985.22,46741360.34,59698211.11,27858.62,58049.68,132950.77,19.88,0.28,True
3757572,6,4240878.05,40771124.53,65569040.72,20345.82,45495.14,52895.83,20.91,0.17,True
3831888,9,12978705.60,45093384.61,66425278.81,12565.30,67073.23,83219.21,19.91,0.28,True


,n_approaches,miss_min,miss_median,miss_max,vel_min,vel_median,vel_max,h,diameter,hazardous
object_id,,,,,,,,,,
2488509,9,12978705.60,45093384.61,66425278.81,12565.30,67073.23,83219.21,20.61,0.20,False
2516803,9,12978705.60,45093384.61,66425278.81,12565.30,67073.23,83219.21,19.52,0.33,False
2529819,8,13765534.89,45154508.84,74781124.91,30420.51,44745.75,74413.76,20.86,0.18,False
3092205,10,10212926.31,39574452.97,60313373.13,25575.96,51312.87,128914.27,19.49,0.34,False
3114023,8,13765534.89,45154508.84,74781124.91,30420.51,44745.75,74413.76,26.10,0.02,False


## Part 3

Question 1)

How many more hazardous and non-hazardas object are there? How many hazardous and non-hazardas observations are there?

In [25]:
n_hazardous = len(hazardous_neos.groupby('object_id'))
n_nonhazardous = len(nonhazardous_neos.groupby('object_id'))
print(f"There are {n_hazardous} hazardous objects and {n_nonhazardous} non-hazardous objects.")

There are 5 hazardous objects and 55 non-hazardous objects.


In [26]:
n_hazardous = len(hazardous_neos)
n_nonhazardous = len(nonhazardous_neos)
print(f"There are {n_hazardous} hazardous observations and {n_nonhazardous} non-hazardous observations.")

There are 41 hazardous observations and 439 non-hazardous observations.
